In [1]:
import torch
import numpy as np
import os
from pathlib import Path
from olmo.config import TrainConfig
from olmo.model import OLMo
from olmo.data import build_train_dataloader
from tqdm import tqdm

In [2]:
# Load all the good stuff
rootdir = Path("/n/netscratch/kempner_sham_lab/Everyone/ameterez/continual_learning/48120231_27/latest-unsharded") # this is best cosine at 32x chinchilla
cfg = TrainConfig.load(rootdir / 'config.yaml')
weights = rootdir / 'model.pt'
olmo_model = OLMo(cfg.model)
# dtype  = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
dtype  = torch.float
device = 'cuda:0'
olmo_model.load_state_dict(torch.load(weights, weights_only=True))
olmo_model = olmo_model.to(device=device, dtype=dtype).eval()

In [3]:
cfg.global_train_batch_size = 1
cfg.data.num_workers = 1
train_loader = build_train_dataloader(cfg)

In [4]:
batch = next(iter(train_loader))
del batch['index']
batch['input_ids'] = batch['input_ids'].cuda()

In [5]:
# torch>=2.0
import math, gc
import torch
from torch.func import functional_call, jvp, vjp

torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_math_sdp(True)

def estimate_alpha_ggn(model, batch, D, M=2, microbatch_size=1, empty_cache_every_mb=True):
    model.eval()

    params  = {n: p.requires_grad_(True) for n, p in model.named_parameters()}
    p_dtype = next(iter(params.values())).dtype
    buffers = {n: (b.to(dtype=p_dtype) if b.is_floating_point() else b)
               for n, b in model.named_buffers()}

    def zeros_like_params(): return {k: torch.zeros_like(v) for k, v in params.items()}
    def add_(a, b):
        for k in a: a[k].add_(b[k])
        return a
    def rad_like():
        return {k: (torch.randint(0,2,v.shape,device=v.device,dtype=torch.int8)*2-1).to(v.dtype)
                for k,v in params.items()}
    def dot(a, b):  return sum((a[k]*b[k]).sum() for k in a)
    def norm2(a):   return sum((a[k]*a[k]).sum() for k in a)

    def split_batch(batch, s, e, B):
        out = {}
        for k, v in batch.items():
            if torch.is_tensor(v) and v.shape[:1] == (B,):
                out[k] = v[s:e]
            else:
                out[k] = v
        return out

    def Hv(vdict):
        B = batch["input_ids"].shape[0]
        out = zeros_like_params()

        for s in range(0, B, microbatch_size):
            e = min(B, s + microbatch_size)
            bmb = split_batch(batch, s, e, B)

            def f(pdict):
                o = functional_call(model, {**pdict, **buffers}, args=(), kwargs=bmb)
                return o.logits[:, :-1, :]  # [b, T-1, V]

            logits, Jv = jvp(f, (params,), (vdict,))

            # analytic CE logit-Hessian: (Diag(p)-p p^T)Jv tokenwise
            p = torch.softmax(logits, dim=-1)
            WJv = p * Jv - (p * Jv).sum(dim=-1, keepdim=True) * p

            _, vjp_fn = vjp(f, params)
            (gmb,) = vjp_fn(WJv)
            add_(out, gmb)

            # aggressively free microbatch intermediates
            del bmb, logits, Jv, p, WJv, gmb, vjp_fn
            if empty_cache_every_mb and torch.cuda.is_available():
                gc.collect()
                torch.cuda.empty_cache()

        return out

    # Hutchinson moments
    m1 = m2 = 0.0
    for _ in range(M):
        v = rad_like()
        u = Hv(v)
        m1 += dot(v, u).item()
        m2 += norm2(u).item()
        del v, u
        if torch.cuda.is_available():
            gc.collect()
            torch.cuda.empty_cache()

    m1 /= M
    m2 /= M
    rhat = m2 / (m1*m1 + 1e-30)

    # solve r(alpha)=rhat where r(alpha)=H_{D,2a}/H_{D,a}^2 is increasing
    def Hsum(s):
        if D <= 200_000:
            i = torch.arange(1, D+1, dtype=torch.float64)
            return float((i.pow(-s)).sum().item())
        if abs(s-1.0) < 1e-8: return 1.0 + math.log(D)
        return 1.0 + (D**(1.0-s) - 1.0) / (1.0 - s)

    def ratio(a):
        h1 = Hsum(a); h2 = Hsum(2.0*a)
        return h2 / (h1*h1)

    lo, hi = 0.0, 10.0
    while ratio(hi) < rhat:
        hi *= 2.0
        if hi > 1e6: break

    for _ in range(80):
        mid = 0.5*(lo+hi)
        if ratio(mid) < rhat: lo = mid
        else: hi = mid

    alpha = 0.5*(lo+hi)
    return alpha, {"m1": m1, "m2": m2, "rhat": rhat}

In [6]:
alpha, info = estimate_alpha_ggn(olmo_model, batch, D=100_000, M=16)
print(alpha, info)

OutOfMemoryError: CUDA out of memory. Tried to allocate 64.00 MiB. GPU 0 has a total capacity of 79.18 GiB of which 49.25 MiB is free. Including non-PyTorch memory, this process has 79.12 GiB memory in use. Of the allocated memory 78.35 GiB is allocated by PyTorch, and 53.90 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)